In [1]:
import pandas as pd
import plotly.graph_objects as go
import os

# 1. DATA INGESTION & DATA INTEGRITY CHECK
# Checking for file existence before loading ensures the pipeline is robust.
file_path = 'expenses.csv'

if not os.path.exists(file_path):
    print("⚠️ Data source not found. Please run the interface (main.py) to initialize the dataset.")
else:
    df = pd.read_csv(file_path)
    
    # 2. CONDITIONAL ANALYTICS GATEWAY
    # This prevents the visualization engine from running on empty datasets.
    if df.empty:
        print("📉 No active capital outflows detected. Please record expenses to generate analytics.")
    else:
        # Pre-processing for time-series accuracy
        df['Date'] = pd.to_datetime(df['Date'])
        df = df.sort_values('Date')

        # --- VISUAL 1: FINANCIAL MOMENTUM (EXPENDITURE VELOCITY INDEX) ---
        fig_trend = go.Figure()

        fig_trend.add_trace(go.Scatter(
            x=df['Date'], y=df['Amount'],
            fill='tozeroy', 
            mode='lines+markers',
            line=dict(width=3, color='#00D4FF', shape='spline'), 
            marker=dict(size=8, color='#00D4FF', line=dict(width=2, color='white')),
            hovertemplate="<b>Period:</b> %{x}<br><b>Outflow:</b> N$%{y:,.2f}<extra></extra>"
        ))

        fig_trend.update_layout(
            title='<b>FINANCIAL MOMENTUM: EXPENDITURE VELOCITY INDEX</b>',
            template='plotly_dark', 
            xaxis_title='Reporting Period',
            yaxis_title='Capital Outflow (N$)',
            font=dict(family="Arial, sans-serif", size=12),
            margin=dict(l=40, r=40, t=60, b=40),
            xaxis=dict(showgrid=False),
            yaxis=dict(gridcolor='#333333', zeroline=False),
            hovermode="x unified"
        )

        fig_trend.show(renderer="notebook_connected", config={'displayModeBar': True})


        # --- VISUAL 2: SECTOR ALLOCATION (ASSET CLASS EXPOSURE) ---
        agg_df = df.groupby('Category')['Amount'].sum().reset_index()

        fig_dist = go.Figure(data=[go.Pie(
            labels=agg_df['Category'], 
            values=agg_df['Amount'], 
            hole=.6,
            pull=[0.15 if amt == agg_df['Amount'].max() else 0 for amt in agg_df['Amount']],
            marker=dict(colors=['#008080', '#20B2AA', '#40E0D0', '#7FFFD4']),
            textinfo='percent+label',
            insidetextorientation='radial',
            hoverinfo='label+value+percent'
        )])

        fig_dist.update_layout(
            title='<b>SECTOR ALLOCATION: ASSET CLASS EXPOSURE</b>',
            template='plotly_dark',
            showlegend=False,
            annotations=[dict(text='CAPITAL<br>DIST', x=0.5, y=0.5, font_size=16, showarrow=False, font_color='white')]
        )

        fig_dist.show(renderer="notebook_connected", config={'displayModeBar': True})
        
        # 3. QUANTITATIVE SUMMARY
        print("--- Portfolio Analysis Complete ---")
        print(f"Total Portfolio Exposure: N${df['Amount'].sum():,.2f}")

--- Portfolio Analysis Complete ---
Total Portfolio Exposure: N$18,300.00
